In [1]:
%run libs.py

In [2]:
%run universal-framework.ipynb

In [3]:
warnings.filterwarnings("ignore")

In [4]:
def sarima_forecast(
    train,
    horizon,
    season_length
):

    try:

        model = SARIMAX(
            train,
            order=(1,1,1),
            seasonal_order=(
                1,
                1,
                1,
                season_length
            ),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fit = model.fit(
            disp=False
        )

        pred = fit.forecast(
            steps=horizon
        )

        return np.array(pred)

    except:

        return np.repeat(
            train[-1],
            horizon
        )

In [ ]:
MAX_SERIES = 100

: 

In [ ]:
results = []

for file in DATA_DIR.glob("*.csv"):

    dataset_name = file.stem

    if dataset_name not in season_lengths:
        continue

    print(
        f"\nRunning SARIMA on {dataset_name}"
    )

    df = pd.read_csv(
        file,
        low_memory=False
    )

    if MAX_SERIES is not None:

        unique_ids = (
            df["unique_id"]
            .drop_duplicates()
            .sample(
                min(
                    MAX_SERIES,
                    df["unique_id"].nunique()
                ),
                random_state=42
            )
        )

        df = df[
            df["unique_id"]
            .isin(unique_ids)
        ]

    season_length = season_lengths[
        dataset_name
    ]

    for horizon, horizon_type in get_horizons_info(
        dataset_name
    ):

        print(
            f"  {horizon_type}"
            f" horizon ({horizon})"
        )

        forecast_fn = (
            lambda train,
                   h,
                   sl=season_length:
            sarima_forecast(
                train,
                h,
                sl
            )
        )

        result = benchmark_dataset(
            df=df,
            horizon=horizon,
            forecast_function=forecast_fn,
            model_name="SARIMA"
        )

        if result is not None:

            result["dataset"] = dataset_name
            result["horizon_type"] = horizon_type
            result["season_length"] = season_length
            result["max_series"] = MAX_SERIES

            results.append(result)

            pd.DataFrame(results).to_csv(
                RESULTS_DIR /
                "sarima.csv",
                index=False
            )

    del df
    gc.collect()

sarima_results = pd.DataFrame(results)


Running SARIMA on std_agri_medium
  short horizon (1)
  medium horizon (3)
  long horizon (5)

Running SARIMA on std_agri_small
  short horizon (1)
  medium horizon (3)
  long horizon (5)

Running SARIMA on std_energy_medium
  short horizon (96)


In [ ]:
display(
    sarima_results.sort_values(
        "MAE"
    )
)